# Parameter Count for exp11.ipynb

In [3]:
%pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-0.30.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (390 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [nbformat]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import math
import nbformat
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
exp11_path = Path('exp11.ipynb')
nb = nbformat.read(exp11_path, as_version=4)
ns = {"torch": torch, "nn": nn, "F": F, "math": math}

for cell in nb.cells:
    if cell.cell_type != 'code':
        continue
    src = cell.source
    if 'class PeriodicSparseAttentionFECG' in src or 'class Conv1DBlock' in src:
        exec(src, ns)

model = ns['CustomApneaModel']()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

Total parameters: 451,874
Trainable parameters: 451,874


In [6]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

class GradCamPlusPlus1D:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def __call__(self, input_tensor, target_time_step=None):
        self.model.zero_grad()
        output = self.model(input_tensor) # Expected shape: [1, 2, SeqLen]
        
        # If target_time_step is None, we explain the point of maximum probability
        if target_time_step is None:
            target_time_step = output[:, 1, :].argmax().item()

        score = output[:, 1, target_time_step]
        score.backward()

        grads = self.gradients # [1, Channels, L']
        activations = self.activations # [1, Channels, L']
        
        # Grad-CAM++ Logic
        grads_power_2 = grads.pow(2)
        grads_power_3 = grads.pow(3)
        
        sum_activations = torch.sum(activations, dim=-1, keepdim=True)
        eps = 1e-7
        aij = grads_power_2 / (2 * grads_power_2 + sum_activations * grads_power_3 + eps)
        
        weights = torch.sum(aij * torch.clamp(grads, min=0), dim=-1, keepdim=True)
        heatmap = torch.sum(weights * activations, dim=1)
        heatmap = torch.clamp(heatmap, min=0)
        
        # Resize heatmap to match input signal length
        heatmap = F.interpolate(heatmap.unsqueeze(0), size=input_tensor.shape[2], mode='linear')
        heatmap = heatmap.squeeze().detach().cpu().numpy()
        
        # Normalize 0-1
        if heatmap.max() > 0:
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
        return heatmap

In [7]:
def plot_gradcam_analysis(signal, heatmap, true_peaks=None, pred_peaks=None, fs=1000, t_start=0, t_end=2):
    """
    signal: (SeqLen,) 1D array
    heatmap: (SeqLen,) 1D array (0-1 normalized)
    """
    time_axis = np.arange(len(signal)) / fs
    i_start, i_end = int(t_start * fs), int(t_end * fs)
    
    # Slice data for the zoom window
    sig_zoom = signal[i_start:i_end]
    heat_zoom = heatmap[i_start:i_end]
    t_zoom = time_axis[i_start:i_end]

    fig, axs = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={'height_ratios': [1, 1]})

    # Top Plot: Signal + Peaks
    axs[0].plot(t_zoom, sig_zoom, color='gray', alpha=0.5, label='AECG Signal')
    if true_peaks is not None:
        tp = true_peaks[(true_peaks >= i_start) & (true_peaks < i_end)]
        axs[0].scatter(tp/fs, signal[tp], color='green', marker='x', s=100, label='True Peak')
    
    axs[0].set_title("Fetal ECG R-Peak Detection Analysis")
    axs[0].legend()

    # Bottom Plot: Heatmap Overlay
    points = np.array([t_zoom, sig_zoom]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    lc = LineCollection(segments, cmap='jet', norm=Normalize(0, 1))
    lc.set_array(heat_zoom)
    lc.set_linewidth(2)
    line = axs[1].add_collection(lc)
    
    axs[1].set_xlim(t_zoom.min(), t_zoom.max())
    axs[1].set_ylim(sig_zoom.min() - 0.5, sig_zoom.max() + 0.5)
    plt.colorbar(line, ax=axs[1], label='Importance (Grad-CAM++)')
    
    plt.tight_layout()
    return fig

In [8]:
# 1. Setup
# Assuming your model has an encoder list or a specific conv layer
# Adjust 'model.encoder[0]' to the last convolutional layer of your specific UNet
target_layer = [m for m in model.modules() if isinstance(m, nn.Conv1d)][-1]
explainer = GradCamPlusPlus1D(model, target_layer)

# 2. Get a sample from your DataLoader
model.eval()
data_iter = iter(test_loader)
signals, labels = next(data_iter) # signals shape: [Batch, Channels, Length]

# 3. Generate Heatmap for the first sample in batch
input_tensor = signals[0:1].to(device)
heatmap = explainer(input_tensor)

# 4. Plot
# Note: Using channel 0 for visualization
signal_to_plot = signals[0, 0, :].cpu().numpy()
fig = plot_gradcam_analysis(signal_to_plot, heatmap, fs=500, t_start=0.5, t_end=2.5)
plt.show()

NameError: name 'test_loader' is not defined